# Parameter Identifiability: dT_scale, perm_frac, advection_scale

Diagnostic notebook for the Tier-3 KO calibration scheme: before spending real IDL/emulator budget on a full campaign over the three active parameters, check cheaply whether they are separable at all -- using a fast analytical proxy model, multi-start local optimization, and a Hessian-based local correlation estimate.

**Read this first.** The model used throughout this notebook is a deliberate SIMPLIFICATION -- a vectorised extension of `icetemp.calibration.physics.cp_model_single` (the same analytical C&P surrogate this codebase already uses for Tier-1/Tier-2 baselines), *not* the real transient IDL forward model. Advection here is represented by a single length-scale rescaling (`z0_eff = z0 / advection_scale`) -- a deliberate simplification, not a solved advection-diffusion PDE. Its *direction* was checked against a real IDL run earlier this session (increasing `advection_scale` from 1.0 to 1.5 measurably COOLED Grenzgletscher's deep profile by ~0.4-0.5 degC; see `01_verify_idl_bridge.ipynb` / project memory), and shrinking `z0_eff` as `advection_scale` grows reproduces that direction (the warm surface anomaly decays faster with depth = colder at depth). So this toy model points the right way, but will not match the real model's magnitude -- it exists to explore parameter-SPACE STRUCTURE cheaply, not to produce calibration-grade numbers. The real forward model for that remains the transient IDL run (`runner.GloGEMRunner`), same as every other surrogate use in this codebase.

**z0 is fixed, not calibrated.** Per the Tier-3 project history, z0 was found to have zero effect on the real transient model (it only shapes the initial spinup profile, which the per-glacier override then overwrites before anything re-reads it) and was replaced by `advection_scale` as the third calibration parameter. It stays fixed here too, at its settings.pro default, purely as the shape parameter the C&P formula needs.

## Cell 1: Setup and Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 -- registers projection='3d'
import seaborn as sns
from scipy import optimize

from icetemp.calibration.data import DataHandler, TARGET_DEPTH_BAND
from icetemp.calibration.priors import Priors, PARAM_NAMES, ADVECTION_SCALE_BOUNDS
from icetemp.calibration.physics import (
    cp_model_single, PERM_FRAC_BOUNDS, DT_SCALE_BOUNDS, ICE_FRAC, Z0_FIRN_DEFAULT,
)

sns.set_theme(style='whitegrid')

print('PARAM_NAMES:', PARAM_NAMES)
print(f'bounds: perm_frac={PERM_FRAC_BOUNDS}  dT_scale={DT_SCALE_BOUNDS}  '
      f'advection_scale={ADVECTION_SCALE_BOUNDS}')
print(f'fixed z0 (settings.pro default, no longer a calibration parameter): {Z0_FIRN_DEFAULT} m')

## Cell 2: Load and Filter Borehole Data

Loads glenglat's CentralEurope calibration set via `DataHandler`, then keeps only entities with at least one observation inside the 10-30 m "thermally settled" band (`TARGET_DEPTH_BAND` -- the same band `data.py` already up-weights by `DEPTH_WEIGHT_IN_BAND=3` for exactly this reason: shallow enough to be well observed, deep enough that the seasonal signal has mostly damped out).

**On the named examples:** Grosser Aletschgletscher has *no* glenglat borehole at all (confirmed by direct search of `borehole.csv` while building the previous notebook -- 0 matches), so it cannot appear here regardless of filtering; that's a fact about the data source, not this filter. Grenzgletscher and Gornergletscher both do have real borehole coverage.

In [ ]:
dh = DataHandler(region='CentralEurope')
dh.load()
dh.summary()

lo, hi = TARGET_DEPTH_BAND
settled = [g for g in dh.calibration_glaciers if np.any((g.depths >= lo) & (g.depths <= hi))]
print(f'\n{len(settled)} / {len(dh.calibration_glaciers)} entities have >=1 observation in the '
      f'{lo:.0f}-{hi:.0f} m band')

named_examples = ['Grosser Aletschgletscher', 'Grenzgletscher', 'Gornergletscher']
print('\nnamed-example coverage:')
for name in named_examples:
    hits = [g for g in settled if g.base_glacier_name == name]
    note = '' if hits else '  <-- not in glenglat at all, or none reach the settled band'
    print(f'  {name:28s}: {len(hits)} settled-band entities{note}')

summary_df = pd.DataFrame([
    {
        'glacier_name': g.glacier_name,
        'base_glacier_name': g.base_glacier_name,
        'elevation_m': g.elevation,
        'regime': 'firn' if g.has_firn_obs and not g.has_ice_obs
                  else ('ice' if g.has_ice_obs and not g.has_firn_obs else 'mixed'),
        'n_obs': g.n_obs,
        'min_depth_m': g.depths.min(),
        'max_depth_m': g.depths.max(),
    }
    for g in settled
]).sort_values('base_glacier_name').reset_index(drop=True)

print(f'\n{len(summary_df)} settled-band entities:')
summary_df

## Cell 3: Fast 1D Analytical Model with Vertical Advection

### The model

Extends `cp_model_single` with one additional term:

```
T(z) = min(0, T_maat + ins * exp(-z / z0_eff))         where   z0_eff = z0 / advection_scale
```

`ins` is the exact same insulation-amplitude term `cp_model_single` already computes:
- firn bands: `ins = dT_scale * dT_firn_band`  (perm_frac has no effect -- matches the real model)
- ice bands:  `ins = perm_frac * ICE_FRAC * dT_scale * dT_firn_band`

and `z0` is held at its fixed settings.pro default (see Cell 1).

**Rationale for the advection term:** stronger vertical/horizontal advection carries colder upglacier/surface ice down faster than diffusion alone can smooth it out, which sharpens the vertical gradient -- equivalent, in this simple exponential-decay form, to a SHORTER effective decay length. `advection_scale=1.5` -> `z0_eff` = 10 m instead of 15 m -> the warm insulation term decays faster with depth -> colder deep ice, steeper gradient. `advection_scale=0.5` -> `z0_eff` = 30 m -> the opposite. This is a deliberate simplification (one rescaled length scale, not a solved advection-diffusion PDE), chosen to be fast enough for the ~150 optimizer runs below.

In [ ]:
def run_analytical_advection_model(depths, dT_scale, perm_frac, advection_scale, glacier_data,
                                    z0=Z0_FIRN_DEFAULT, is_firn=None):
    """Vectorised C&P-plus-advection profile (see markdown above).

    is_firn: per-depth regime flags. Defaults to glacier_data.is_firn when `depths` matches its
    own observation depths exactly (the FITTING use case, Cells 5-6 -- each observation keeps its
    own real regime); otherwise falls back to glacier_data's majority regime as one flag for the
    whole curve (the smooth-PLOTTING use case below, where `depths` is an arbitrary fine grid
    with no per-point regime information of its own).
    """
    depths = np.asarray(depths, dtype=float)
    if is_firn is None:
        if len(glacier_data.is_firn) == len(depths):
            is_firn = glacier_data.is_firn
        else:
            is_firn = np.mean(glacier_data.is_firn) >= 0.5
    is_firn = np.broadcast_to(np.asarray(is_firn, dtype=bool), depths.shape)

    ins_firn = dT_scale * glacier_data.dT_firn_band
    ins_ice = perm_frac * ICE_FRAC * dT_scale * glacier_data.dT_firn_band
    ins = np.where(is_firn, ins_firn, ins_ice)

    z0_eff = z0 / advection_scale
    T = glacier_data.T_maat + ins * np.exp(-depths / z0_eff)
    return np.minimum(0.0, T)

In [ ]:
plot_glacier = max(settled, key=lambda g: g.n_obs)  # most-observed entity -> clearest illustration
print(f'plotting: {plot_glacier.glacier_name}  (n_obs={plot_glacier.n_obs}, '
      f'depth range {plot_glacier.depths.min():.0f}-{plot_glacier.depths.max():.0f} m, '
      f'regime={"firn" if plot_glacier.has_firn_obs else "ice"})')

z_smooth = np.linspace(0.5, max(plot_glacier.depths.max(), 40), 200)
pf0, ds0 = 0.8, 1.2   # arbitrary but fixed surface/percolation params -- isolate advection's effect

fig, ax = plt.subplots(figsize=(6, 8))
ax.scatter(plot_glacier.T_obs, plot_glacier.depths,
           s=90 * plot_glacier.weights / plot_glacier.weights.max(),
           color='black', zorder=5, label=f'{plot_glacier.glacier_name} observations')

for adv, color, label in [(0.5, '#DD8452', 'under-advection (0.5, warmer deep)'),
                           (1.0, '#4C72B0', 'baseline (1.0, unscaled)'),
                           (1.5, '#C44E52', 'over-advection (1.5, colder deep, steep)')]:
    T_curve = run_analytical_advection_model(z_smooth, ds0, pf0, adv, plot_glacier)
    ax.plot(T_curve, z_smooth, color=color, lw=2, label=label)

ax.invert_yaxis()
ax.set_xlabel('Temperature [\u00b0C]')
ax.set_ylabel('Depth [m]')
ax.set_title(f'{plot_glacier.glacier_name}: toy advection-C&P model vs observations\n'
             f'(perm_frac={pf0}, dT_scale={ds0} held fixed across all three curves)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Cell 4: Multi-Start Optimization Setup

A local optimizer (L-BFGS-B, chosen over Nelder-Mead for its native bounds support and faster convergence) finds whichever minimum is nearest its starting point -- it says nothing about the cost surface's global shape from a single run. Launching many runs from well-dispersed starting points and looking at where they CONVERGE reveals that shape cheaply: a tight cluster of endpoints means a well-identified minimum (every start finds the same answer); a spread-out ridge or line means a family of equally-good solutions (non-identifiability).

In [ ]:
N_STARTS = 50
rng = np.random.default_rng(42)

bounds_3d = [PERM_FRAC_BOUNDS, DT_SCALE_BOUNDS, ADVECTION_SCALE_BOUNDS]
starts_3d = np.column_stack([rng.uniform(b[0], b[1], size=N_STARTS) for b in bounds_3d])

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(starts_3d[:, 0], starts_3d[:, 1], starts_3d[:, 2], c='#4C72B0', s=30, alpha=0.8)
ax.set_xlabel('perm_frac')
ax.set_ylabel('dT_scale')
ax.set_zlabel('advection_scale')
ax.set_title(f'{N_STARTS} multi-start initial points (uniform random over the prior bounds)')
plt.tight_layout()
plt.show()

print(f'perm_frac        sampled range: [{starts_3d[:,0].min():.3f}, {starts_3d[:,0].max():.3f}]  '
      f'(bounds {PERM_FRAC_BOUNDS})')
print(f'dT_scale         sampled range: [{starts_3d[:,1].min():.3f}, {starts_3d[:,1].max():.3f}]  '
      f'(bounds {DT_SCALE_BOUNDS})')
print(f'advection_scale  sampled range: [{starts_3d[:,2].min():.3f}, {starts_3d[:,2].max():.3f}]  '
      f'(bounds {ADVECTION_SCALE_BOUNDS})')

## Cell 5: Case A -- The Complete 3D Parameter Space

Fits all three parameters jointly against `plot_glacier` from Cell 3 (the most-observed settled-band entity -- a data-driven choice, not cherry-picked for a particular outcome: whichever entity has the most points is, mechanically, the best-conditioned one for constraining 3 unknowns).

In [ ]:
def make_cost_fn_3d(glacier):
    """Weighted RMSE, using glacier.weights (DEPTH_WEIGHT_IN_BAND up-weighting, same convention
    as the rest of this codebase's grid search / KO likelihood)."""
    def cost(theta):
        pf, ds, adv = theta
        T_mod = run_analytical_advection_model(glacier.depths, ds, pf, adv, glacier)
        resid = T_mod - glacier.T_obs
        return np.sqrt(np.average(resid ** 2, weights=glacier.weights))
    return cost


fit_glacier = plot_glacier   # keep it consistent with the Cell 3 illustration
cost_a = make_cost_fn_3d(fit_glacier)

results_a = [optimize.minimize(cost_a, x0, method='L-BFGS-B', bounds=bounds_3d) for x0 in starts_3d]

converged_a = np.array([r.x for r in results_a if r.success])
final_cost_a = np.array([r.fun for r in results_a if r.success])
print(f'Case A ({fit_glacier.glacier_name}): {len(converged_a)}/{N_STARTS} runs converged '
      f'(scipy success flag)')
print(f'final RMSE across runs: min={final_cost_a.min():.4f}  max={final_cost_a.max():.4f}  '
      f'median={np.median(final_cost_a):.4f}')

best_a = results_a[int(np.argmin([r.fun for r in results_a]))]
print(f'best-of-{N_STARTS}: perm_frac={best_a.x[0]:.4f}  dT_scale={best_a.x[1]:.4f}  '
      f'advection_scale={best_a.x[2]:.4f}  RMSE={best_a.fun:.4f}')

converged_a_df = pd.DataFrame(converged_a, columns=['perm_frac', 'dT_scale', 'advection_scale'])

## Cell 6: Case B -- Bare Ice Tongue (Firn Absent) Trade-Off

On a pure-ablation (bare-ice) entity, `cp_model_single`'s own ice-band formula is `ins = perm_frac * ICE_FRAC * dT_scale * dT_firn_band` -- perm_frac and dT_scale enter ONLY as a product. Any (perm_frac, dT_scale) pair with the same product gives IDENTICAL model output at every depth, so the optimizer cannot tell them apart: a whole hyperbola `perm_frac * dT_scale = const` is equally good, not a single point. `advection_scale` is held fixed here specifically to isolate this ridge cleanly in 2D (with it free too, the same ridge would still exist, just embedded one dimension higher and harder to see directly).

In [ ]:
bare_ice = [g for g in settled if g.has_ice_obs and not g.has_firn_obs]
print(f'{len(bare_ice)} entities are pure bare-ice (ablation-only) in the settled-band set:')
for g in sorted(bare_ice, key=lambda g: -g.n_obs)[:10]:
    print(f'  {g.glacier_name:30s}  n_obs={g.n_obs}  depth {g.depths.min():.0f}-{g.depths.max():.0f} m')

assert bare_ice, 'no pure bare-ice entity in the settled-band set -- widen the region/filters'
ice_glacier = max(bare_ice, key=lambda g: g.n_obs)
print(f'\nusing: {ice_glacier.glacier_name}')

FIXED_ADV = 1.0   # advection_scale held fixed -- see markdown: isolates the perm_frac*dT_scale ridge


def make_cost_fn_2d(glacier, fixed_advection_scale=FIXED_ADV):
    def cost(theta):
        pf, ds = theta
        T_mod = run_analytical_advection_model(glacier.depths, ds, pf, fixed_advection_scale, glacier)
        resid = T_mod - glacier.T_obs
        return np.sqrt(np.average(resid ** 2, weights=glacier.weights))
    return cost


bounds_2d = [PERM_FRAC_BOUNDS, DT_SCALE_BOUNDS]
starts_2d = starts_3d[:, :2]   # reuse the SAME 50 (perm_frac, dT_scale) starts as Case A

cost_b = make_cost_fn_2d(ice_glacier)
results_b = [optimize.minimize(cost_b, x0, method='L-BFGS-B', bounds=bounds_2d) for x0 in starts_2d]

converged_b = np.array([r.x for r in results_b if r.success])
final_cost_b = np.array([r.fun for r in results_b if r.success])
print(f'Case B ({ice_glacier.glacier_name}): {len(converged_b)}/{N_STARTS} runs converged')
print(f'final RMSE across runs: min={final_cost_b.min():.4f}  max={final_cost_b.max():.4f}  '
      f'median={np.median(final_cost_b):.4f}')

converged_b_df = pd.DataFrame(converged_b, columns=['perm_frac', 'dT_scale'])
converged_b_df['product_pf_ds'] = converged_b_df['perm_frac'] * converged_b_df['dT_scale']
print(f"perm_frac * dT_scale across the {len(converged_b_df)} endpoints: "
      f"mean={converged_b_df['product_pf_ds'].mean():.4f}  std={converged_b_df['product_pf_ds'].std():.4f}  "
      f"(a SMALL std here despite widely spread individual perm_frac/dT_scale values IS the ridge)")

## Cell 7: Pairwise Corner Plots

Panel 1 (Case A) vs Panel 2 (Case B), then a full pairwise grid for Case A's three parameters.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

sc = axes[0].scatter(converged_a_df['perm_frac'], converged_a_df['dT_scale'],
                      c=converged_a_df['advection_scale'], cmap='viridis', s=40,
                      edgecolor='k', linewidth=0.3)
axes[0].set_xlabel('perm_frac')
axes[0].set_ylabel('dT_scale')
axes[0].set_title(f'Case A ({fit_glacier.glacier_name})\n{len(converged_a_df)} converged endpoints')
plt.colorbar(sc, ax=axes[0], label='advection_scale')

axes[1].scatter(converged_b_df['perm_frac'], converged_b_df['dT_scale'], color='#C44E52', s=40,
                 edgecolor='k', linewidth=0.3)
pf_line = np.linspace(*PERM_FRAC_BOUNDS, 100)
axes[1].plot(pf_line, converged_b_df['product_pf_ds'].median() / pf_line, 'k--', lw=1,
             label=f'perm_frac * dT_scale = {converged_b_df["product_pf_ds"].median():.3f} (median)')
axes[1].set_xlabel('perm_frac')
axes[1].set_ylabel('dT_scale')
axes[1].set_ylim(*DT_SCALE_BOUNDS)
axes[1].set_title(f'Case B ({ice_glacier.glacier_name}, bare ice, advection_scale fixed={FIXED_ADV})\n'
                   f'{len(converged_b_df)} converged endpoints')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
pg = sns.PairGrid(converged_a_df, diag_sharey=False)
pg.map_lower(sns.scatterplot, s=25, alpha=0.7)
pg.map_diag(sns.histplot, kde=True)
pg.map_upper(sns.scatterplot, s=25, alpha=0.7)
pg.fig.suptitle(f'Case A full pairwise structure ({fit_glacier.glacier_name})', y=1.02)
plt.show()

**How to read these.** A tight, roughly circular/elliptical cluster (Case A) means the optimizer lands on essentially the same answer regardless of starting point -- well identified. A stretched diagonal band or hyperbola (Case B) means many different (perm_frac, dT_scale) pairs fit equally well -- non-identifiability, visible directly rather than inferred. Whatever these actually show for this real data is the honest result to report -- with real, often sparse, glenglat depth coverage there is no guarantee Case A comes out perfectly tight; a partially elongated cluster is itself useful information, not a failure of the notebook.

## Cell 8: Mathematical Correlation Matrix (The Hessian)

For a Gaussian-error likelihood with chi-square objective χ²(θ) = Σᵢ ((T_mod,ᵢ(θ) − T_obs,ᵢ) / σᵢ)², the Laplace approximation says the posterior near its mode θ̂ is approximately Gaussian with

```
Cov(theta)  ~  [ Hessian of (0.5 * chi^2) at theta_hat ]^-1
```

i.e. the inverse of the Hessian of the (half) chi-square objective at the optimum -- the same object as the observed Fisher information matrix. Off-diagonal correlations rho_ij = Cov_ij / sqrt(Cov_ii * Cov_jj) close to +/-1 mean the data cannot locally separate parameter i from parameter j -- the same non-identifiability the corner plots show above, but as one number per pair instead of a picture.

**Note the objective switch:** this uses `sigma` (real per-observation glenglat uncertainty) and a proper chi-square, not the plain weighted-RMSE used to DRIVE the optimizer in Cells 5-6. RMSE is a perfectly good thing to minimize, but its Hessian is not a valid covariance estimate; chi-square with real observation errors is what the Laplace approximation actually requires.

In [ ]:
def numerical_hessian(f, x0, h=1e-3):
    """Central-difference Hessian of scalar function f at x0. h is an ABSOLUTE step per
    dimension -- fine here since all three parameters are O(1) in the same rough range; would
    need per-dimension scaling for parameters spanning orders of magnitude."""
    n = len(x0)
    H = np.zeros((n, n))
    f0 = f(x0)
    for i in range(n):
        for j in range(n):
            if i == j:
                xp = x0.copy(); xp[i] += h
                xm = x0.copy(); xm[i] -= h
                H[i, i] = (f(xp) - 2 * f0 + f(xm)) / h ** 2
            elif j > i:
                xpp = x0.copy(); xpp[i] += h; xpp[j] += h
                xpm = x0.copy(); xpm[i] += h; xpm[j] -= h
                xmp = x0.copy(); xmp[i] -= h; xmp[j] += h
                xmm = x0.copy(); xmm[i] -= h; xmm[j] -= h
                H[i, j] = H[j, i] = (f(xpp) - f(xpm) - f(xmp) + f(xmm)) / (4 * h ** 2)
    return H


def chi2_3d(theta, glacier):
    pf, ds, adv = theta
    T_mod = run_analytical_advection_model(glacier.depths, ds, pf, adv, glacier)
    return np.sum(((T_mod - glacier.T_obs) / glacier.sigma) ** 2)


theta_hat = np.array([best_a.x[0], best_a.x[1], best_a.x[2]])
half_chi2 = lambda theta: 0.5 * chi2_3d(theta, fit_glacier)

H = numerical_hessian(half_chi2, theta_hat)
print('Hessian of 0.5*chi^2 at the Case A optimum:')
print(np.round(H, 4))

corr = None
try:
    cov = np.linalg.inv(H)
    if np.any(np.diag(cov) < 0):
        raise np.linalg.LinAlgError('negative variance -- Hessian not positive definite here')
    sd = np.sqrt(np.diag(cov))
    corr = cov / np.outer(sd, sd)
except np.linalg.LinAlgError as e:
    print(f'WARNING: could not invert the Hessian into a valid covariance ({e}). This is itself '
          f'diagnostic -- it means the optimum is not a clean local minimum in all 3 directions '
          f'(e.g. sitting on or near a ridge, or pinned against a parameter bound).')

if corr is not None:
    param_names = ['perm_frac', 'dT_scale', 'advection_scale']
    corr_df = pd.DataFrame(corr, index=param_names, columns=param_names)

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(corr_df, annot=True, fmt='.3f', vmin=-1, vmax=1, cmap='coolwarm', ax=ax,
                cbar_kws={'label': 'correlation'})
    ax.set_title(f'Local parameter correlation matrix\n(Case A optimum, {fit_glacier.glacier_name})')
    plt.tight_layout()
    plt.show()

    print('\nStandard errors (1-sigma, local/asymptotic):')
    for name, s in zip(param_names, sd):
        print(f'  {name:16s} {s:.4f}')

    print('\nOff-diagonal correlations exceeding +/-0.8 (data cannot locally separate these):')
    flagged = False
    for i in range(3):
        for j in range(i + 1, 3):
            if abs(corr[i, j]) > 0.8:
                flagged = True
                print(f'  {param_names[i]} <-> {param_names[j]}: rho = {corr[i, j]:.3f}')
    if not flagged:
        print('  none -- all three parameters are locally separable at this optimum.')